# k-Nearest Neighbours (kNN) Model Selection

This notebook evaluates the 10-fold cross-validation grid search for the **Lazy Learning / kNN** classifier across two benchmark datasets:
- **Soybean** (683 instances, 35 nominal features)
- **Hypothyroid** (3,772 instances, 29 mixed nominal and continuous features)

### Methodology (Demšar, 2006)
In accordance with sound statistical comparison protocols, candidate configurations are evaluated as complete models rather than isolating each individual parameter.

For each dataset, we conduct two distinct, complementary model selections:
1. **Accuracy-Oriented Selection**: Identifies the configuration with the highest predictive accuracy across folds (tie-breaking with prediction efficiency).
2. **Efficiency-Oriented Selection**: Identifies the configuration that minimizes prediction latency (inference time in milliseconds) while retaining competitive accuracy.

Both selections apply Demšar's methodology:
- Friedman Omnibus test across candidate configurations evaluated over $N=10$ independent folds
- Critical Difference (CD) diagram for statistically equivalent cliques
- Wilcoxon Signed-Rank Test between rank #1 and rank #2


In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Setup paths dynamically
current_dir = Path().resolve()
if (current_dir / "Work2").exists():
    ROOT = current_dir / "Work2"
elif current_dir.name == "analysis":
    ROOT = current_dir.parent
else:
    ROOT = current_dir
sys.path.insert(0, str(ROOT))

from analysis.statistical_utils import (
    load_fold_results,
    load_summary_results,
    run_friedman_analysis,
    select_best_configuration,
    RESULTS_DIR
)
from KNNClassifier import (
    friedman_test,
    wilcoxon_test,
    plot_critical_difference_diagram
)

print(f"Project root: {ROOT}")
print(f"Results directory: {RESULTS_DIR}")


Project root: C:\Users\Administrator\IML-Lab-2026\Work2
Results directory: C:\Users\Administrator\IML-Lab-2026\Work2\results


## 1. Soybean Dataset: kNN Model Selection


In [2]:
# Dynamically load Soybean results
df_soy_folds = load_fold_results("knn", "soybean")
df_soy_summary = load_summary_results("knn", "soybean")

print(f"Loaded {len(df_soy_folds)} fold evaluations across {len(df_soy_summary)} distinct configurations.")


Loaded 960 fold evaluations across 96 distinct configurations.


### 1.1 Soybean: Best kNN Model Selected by Accuracy


In [3]:
# 1. Best Configuration Selected by Accuracy
print("Top 10 Configurations by Mean Accuracy (Soybean):")
display(df_soy_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "k", "distance_metric", "voting_scheme", "retention_policy", 
     "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_best_soy_acc = select_best_configuration(
    df_soy_folds, df_soy_summary, dataset="soybean", algo_name="KNN", metric="accuracy"
)

print("\n=== Selected Best kNN Model by Accuracy (Soybean) ===")
print(f"  Model:                 {res_best_soy_acc['best_model']}")
print(f"  Accuracy:              {res_best_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_best_soy_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_best_soy_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Storage Retained:      {res_best_soy_acc['meta']['mean_storage_pct']:.1f}%")
print(f"  Friedman Chi2 (acc):   {res_best_soy_acc['meta']['friedman_chi2']:.4f} (p-value: {res_best_soy_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_best_soy_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 Configurations by Mean Accuracy (Soybean):


,model,k,distance_metric,voting_scheme,retention_policy,accuracy,accuracy_std,prediction_time_ms
0,"KNN(k=3, metric=canberra, voting=majority, retention=always_retain)",3,canberra,majority,always_retain,0.926139,0.026553,52.62837
1,"KNN(k=3, metric=canberra, voting=inverse_distance, retention=always_retain)",3,canberra,inverse_distance,always_retain,0.924646,0.028791,66.07050
2,"KNN(k=7, metric=manhattan, voting=majority, retention=different_class)",7,manhattan,majority,different_class,0.923394,0.030279,23.40304
5,"KNN(k=7, metric=euclidean, voting=majority, retention=different_class)",7,euclidean,majority,different_class,0.923394,0.030279,23.92616
3,"KNN(k=7, metric=heom, voting=majority, retention=different_class)",7,heom,majority,different_class,0.923394,0.030279,38.14954
4,"KNN(k=7, metric=canberra, voting=majority, retention=different_class)",7,canberra,majority,different_class,0.923394,0.030279,47.51167
6,"KNN(k=3, metric=euclidean, voting=majority, retention=always_retain)",3,euclidean,majority,always_retain,0.923190,0.026382,27.19510
8,"KNN(k=3, metric=manhattan, voting=majority, retention=always_retain)",3,manhattan,majority,always_retain,0.923190,0.026382,27.69953
7,"KNN(k=3, metric=heom, voting=majority, retention=always_retain)",3,heom,majority,always_retain,0.923190,0.026382,44.70203
9,"KNN(k=7, metric=manhattan, voting=inverse_distance, retention=different_class)",7,manhattan,inverse_distance,different_class,0.921715,0.028154,31.30260



=== Selected Best kNN Model by Accuracy (Soybean) ===
  Model:                 KNN(k=3, metric=canberra, voting=majority, retention=always_retain)
  Accuracy:              92.61% ± 2.66%
  Prediction Time:       52.63 ms
  Storage Retained:      111.1%
  Friedman Chi2 (acc):   143.0028 (p-value: 1.0655e-03)
  Wilcoxon #1 vs #2 (p): 1.0000


### 1.2 Soybean: Best kNN Model Selected by Efficiency (Prediction Latency)


In [4]:
# 2. Best Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 Configurations by Efficiency / Prediction Time (Soybean):")
display(df_soy_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "k", "distance_metric", "voting_scheme", "retention_policy", 
     "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_best_soy_eff = select_best_configuration(
    df_soy_folds, df_soy_summary, dataset="soybean", algo_name="KNN", metric="prediction_time"
)

print("\n=== Selected Best kNN Model by Efficiency (Soybean) ===")
print(f"  Model:                 {res_best_soy_eff['best_model']}")
print(f"  Accuracy:              {res_best_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_best_soy_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_best_soy_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Storage Retained:      {res_best_soy_eff['meta']['mean_storage_pct']:.1f}%")
print(f"  Friedman Chi2 (time):  {res_best_soy_eff['meta']['friedman_chi2']:.4f} (p-value: {res_best_soy_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_best_soy_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 Configurations by Efficiency / Prediction Time (Soybean):


,model,k,distance_metric,voting_scheme,retention_policy,accuracy,accuracy_std,prediction_time_ms
56,"KNN(k=5, metric=euclidean, voting=majority, retention=never_retain)",5,euclidean,majority,never_retain,0.915719,0.031551,18.08878
18,"KNN(k=3, metric=manhattan, voting=majority, retention=never_retain)",3,manhattan,majority,never_retain,0.921456,0.031196,18.34137
24,"KNN(k=7, metric=manhattan, voting=majority, retention=never_retain)",7,manhattan,majority,never_retain,0.920280,0.027592,18.98428
21,"KNN(k=3, metric=euclidean, voting=majority, retention=never_retain)",3,euclidean,majority,never_retain,0.921456,0.031196,19.74823
27,"KNN(k=7, metric=euclidean, voting=majority, retention=never_retain)",7,euclidean,majority,never_retain,0.920280,0.027592,20.57528
55,"KNN(k=5, metric=manhattan, voting=majority, retention=never_retain)",5,manhattan,majority,never_retain,0.915719,0.031551,21.14686
51,"KNN(k=3, metric=manhattan, voting=majority, retention=different_class)",3,manhattan,majority,different_class,0.915746,0.033743,21.97897
50,"KNN(k=3, metric=euclidean, voting=majority, retention=different_class)",3,euclidean,majority,different_class,0.915746,0.033743,22.15643
58,"KNN(k=3, metric=manhattan, voting=majority, retention=oblivion_by_goodness)",3,manhattan,majority,oblivion_by_goodness,0.914791,0.037885,22.42981
63,"KNN(k=3, metric=euclidean, voting=majority, retention=oblivion_by_goodness)",3,euclidean,majority,oblivion_by_goodness,0.914791,0.037885,22.93282



=== Selected Best kNN Model by Efficiency (Soybean) ===
  Model:                 KNN(k=5, metric=euclidean, voting=majority, retention=never_retain)
  Accuracy:              91.57% ± 3.16%
  Prediction Time:       18.09 ms
  Storage Retained:      100.0%
  Friedman Chi2 (time):  896.4521 (p-value: 1.2720e-130)
  Wilcoxon #1 vs #2 (p): 0.8457


## 2. Hypothyroid Dataset: kNN Model Selection


In [5]:
# Dynamically load Hypothyroid results
df_hypo_folds = load_fold_results("knn", "hypothyroid")
df_hypo_summary = load_summary_results("knn", "hypothyroid")

print(f"Loaded {len(df_hypo_folds)} fold evaluations across {len(df_hypo_summary)} distinct configurations.")


Loaded 960 fold evaluations across 96 distinct configurations.


### 2.1 Hypothyroid: Best kNN Model Selected by Accuracy


In [6]:
# 1. Best Configuration Selected by Accuracy
print("Top 10 Configurations by Mean Accuracy (Hypothyroid):")
display(df_hypo_summary.sort_values(by=["accuracy", "prediction_time"], ascending=[False, True])[
    ["model", "k", "distance_metric", "voting_scheme", "retention_policy", 
     "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_best_hypo_acc = select_best_configuration(
    df_hypo_folds, df_hypo_summary, dataset="hypothyroid", algo_name="KNN", metric="accuracy"
)

print("\n=== Selected Best kNN Model by Accuracy (Hypothyroid) ===")
print(f"  Model:                 {res_best_hypo_acc['best_model']}")
print(f"  Accuracy:              {res_best_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_best_hypo_acc['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_best_hypo_acc['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Storage Retained:      {res_best_hypo_acc['meta']['mean_storage_pct']:.1f}%")
print(f"  Friedman Chi2 (acc):   {res_best_hypo_acc['meta']['friedman_chi2']:.4f} (p-value: {res_best_hypo_acc['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_best_hypo_acc['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 Configurations by Mean Accuracy (Hypothyroid):


,model,k,distance_metric,voting_scheme,retention_policy,accuracy,accuracy_std,prediction_time_ms
0,"KNN(k=5, metric=canberra, voting=inverse_distance, retention=always_retain)",5,canberra,inverse_distance,always_retain,0.966343,0.010646,892.78572
1,"KNN(k=5, metric=canberra, voting=inverse_distance, retention=never_retain)",5,canberra,inverse_distance,never_retain,0.966078,0.010843,661.00017
2,"KNN(k=5, metric=canberra, voting=inverse_distance, retention=different_class)",5,canberra,inverse_distance,different_class,0.966078,0.010988,687.07495
3,"KNN(k=3, metric=canberra, voting=inverse_distance, retention=different_class)",3,canberra,inverse_distance,different_class,0.965813,0.011041,702.17146
4,"KNN(k=3, metric=canberra, voting=majority, retention=different_class)",3,canberra,majority,different_class,0.965285,0.011985,656.18743
5,"KNN(k=3, metric=canberra, voting=inverse_distance, retention=never_retain)",3,canberra,inverse_distance,never_retain,0.965282,0.010970,664.38192
6,"KNN(k=5, metric=canberra, voting=majority, retention=always_retain)",5,canberra,majority,always_retain,0.964754,0.010037,856.11746
7,"KNN(k=3, metric=canberra, voting=inverse_distance, retention=always_retain)",3,canberra,inverse_distance,always_retain,0.964754,0.009965,933.81548
8,"KNN(k=3, metric=canberra, voting=majority, retention=never_retain)",3,canberra,majority,never_retain,0.964754,0.011894,629.01148
9,"KNN(k=3, metric=canberra, voting=inverse_distance, retention=oblivion_by_goodness)",3,canberra,inverse_distance,oblivion_by_goodness,0.964490,0.011768,699.41838



=== Selected Best kNN Model by Accuracy (Hypothyroid) ===
  Model:                 KNN(k=5, metric=canberra, voting=inverse_distance, retention=always_retain)
  Accuracy:              96.63% ± 1.06%
  Prediction Time:       892.79 ms
  Storage Retained:      111.1%
  Friedman Chi2 (acc):   670.6383 (p-value: 1.9756e-87)
  Wilcoxon #1 vs #2 (p): 1.0000


### 2.2 Hypothyroid: Best kNN Model Selected by Efficiency (Prediction Latency)


In [7]:
# 2. Best Configuration Selected by Efficiency (Lowest Prediction Time)
print("Top 10 Configurations by Efficiency / Prediction Time (Hypothyroid):")
display(df_hypo_summary.sort_values(by=["prediction_time", "accuracy"], ascending=[True, False])[
    ["model", "k", "distance_metric", "voting_scheme", "retention_policy", 
     "accuracy", "accuracy_std", "prediction_time_ms"]
].head(10))

res_best_hypo_eff = select_best_configuration(
    df_hypo_folds, df_hypo_summary, dataset="hypothyroid", algo_name="KNN", metric="prediction_time"
)

print("\n=== Selected Best kNN Model by Efficiency (Hypothyroid) ===")
print(f"  Model:                 {res_best_hypo_eff['best_model']}")
print(f"  Accuracy:              {res_best_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_best_hypo_eff['meta']['std_accuracy']*100:.2f}%")
print(f"  Prediction Time:       {res_best_hypo_eff['meta']['mean_prediction_time_ms']:.2f} ms")
print(f"  Storage Retained:      {res_best_hypo_eff['meta']['mean_storage_pct']:.1f}%")
print(f"  Friedman Chi2 (time):  {res_best_hypo_eff['meta']['friedman_chi2']:.4f} (p-value: {res_best_hypo_eff['meta']['friedman_p']:.4e})")
print(f"  Wilcoxon #1 vs #2 (p): {res_best_hypo_eff['meta']['wilcoxon_best_vs_second_p']:.4f}")


Top 10 Configurations by Efficiency / Prediction Time (Hypothyroid):


,model,k,distance_metric,voting_scheme,retention_policy,accuracy,accuracy_std,prediction_time_ms
47,"KNN(k=5, metric=manhattan, voting=majority, retention=never_retain)",5,manhattan,majority,never_retain,0.934266,0.007760,265.59645
48,"KNN(k=7, metric=euclidean, voting=majority, retention=never_retain)",7,euclidean,majority,never_retain,0.934262,0.006112,267.21235
40,"KNN(k=7, metric=manhattan, voting=majority, retention=never_retain)",7,manhattan,majority,never_retain,0.935061,0.007706,269.98726
57,"KNN(k=3, metric=manhattan, voting=majority, retention=never_retain)",3,manhattan,majority,never_retain,0.933466,0.005547,271.49354
62,"KNN(k=5, metric=euclidean, voting=majority, retention=never_retain)",5,euclidean,majority,never_retain,0.932940,0.007690,273.11541
61,"KNN(k=3, metric=euclidean, voting=majority, retention=never_retain)",3,euclidean,majority,never_retain,0.933205,0.008265,292.11263
54,"KNN(k=3, metric=manhattan, voting=majority, retention=oblivion_by_goodness)",3,manhattan,majority,oblivion_by_goodness,0.933995,0.005264,298.38063
43,"KNN(k=7, metric=euclidean, voting=majority, retention=different_class)",7,euclidean,majority,different_class,0.934793,0.007100,304.24445
51,"KNN(k=5, metric=manhattan, voting=majority, retention=oblivion_by_goodness)",5,manhattan,majority,oblivion_by_goodness,0.933998,0.006665,304.32386
29,"KNN(k=5, metric=manhattan, voting=inverse_distance, retention=never_retain)",5,manhattan,inverse_distance,never_retain,0.936125,0.009608,305.53972



=== Selected Best kNN Model by Efficiency (Hypothyroid) ===
  Model:                 KNN(k=5, metric=manhattan, voting=majority, retention=never_retain)
  Accuracy:              93.43% ± 0.78%
  Prediction Time:       265.60 ms
  Storage Retained:      100.0%
  Friedman Chi2 (time):  926.4438 (p-value: 1.7982e-136)
  Wilcoxon #1 vs #2 (p): 0.6953


## 3. Dynamically Generated Summary: Best kNN Configurations


In [8]:
# Compile summary dynamically from the selected models
summary_rows = [
    {
        "Dataset": "Soybean",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_best_soy_acc["best_model"],
        "Accuracy (%)": f"{res_best_soy_acc['meta']['mean_accuracy']*100:.2f}% ± {res_best_soy_acc['meta']['std_accuracy']*100:.2f}%",
        "Prediction Time (ms)": f"{res_best_soy_acc['meta']['mean_prediction_time_ms']:.2f}",
        "Storage Retained (%)": f"{res_best_soy_acc['meta']['mean_storage_pct']:.1f}%",
    },
    {
        "Dataset": "Soybean",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_best_soy_eff["best_model"],
        "Accuracy (%)": f"{res_best_soy_eff['meta']['mean_accuracy']*100:.2f}% ± {res_best_soy_eff['meta']['std_accuracy']*100:.2f}%",
        "Prediction Time (ms)": f"{res_best_soy_eff['meta']['mean_prediction_time_ms']:.2f}",
        "Storage Retained (%)": f"{res_best_soy_eff['meta']['mean_storage_pct']:.1f}%",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Accuracy",
        "Best Configuration": res_best_hypo_acc["best_model"],
        "Accuracy (%)": f"{res_best_hypo_acc['meta']['mean_accuracy']*100:.2f}% ± {res_best_hypo_acc['meta']['std_accuracy']*100:.2f}%",
        "Prediction Time (ms)": f"{res_best_hypo_acc['meta']['mean_prediction_time_ms']:.2f}",
        "Storage Retained (%)": f"{res_best_hypo_acc['meta']['mean_storage_pct']:.1f}%",
    },
    {
        "Dataset": "Hypothyroid",
        "Criterion": "Best Efficiency",
        "Best Configuration": res_best_hypo_eff["best_model"],
        "Accuracy (%)": f"{res_best_hypo_eff['meta']['mean_accuracy']*100:.2f}% ± {res_best_hypo_eff['meta']['std_accuracy']*100:.2f}%",
        "Prediction Time (ms)": f"{res_best_hypo_eff['meta']['mean_prediction_time_ms']:.2f}",
        "Storage Retained (%)": f"{res_best_hypo_eff['meta']['mean_storage_pct']:.1f}%",
    },
]

df_best_knn_summary = pd.DataFrame(summary_rows)
print("=== Final Selected Best kNN Models Summary (Accuracy vs Efficiency) ===")
display(df_best_knn_summary)


=== Final Selected Best kNN Models Summary (Accuracy vs Efficiency) ===


,Dataset,Criterion,Best Configuration,Accuracy (%),Prediction Time (ms),Storage Retained (%)
0,Soybean,Best Accuracy,"KNN(k=3, metric=canberra, voting=majority, retention=always_retain)",92.61% ± 2.66%,52.63,111.1%
1,Soybean,Best Efficiency,"KNN(k=5, metric=euclidean, voting=majority, retention=never_retain)",91.57% ± 3.16%,18.09,100.0%
2,Hypothyroid,Best Accuracy,"KNN(k=5, metric=canberra, voting=inverse_distance, retention=always_retain)",96.63% ± 1.06%,892.79,111.1%
3,Hypothyroid,Best Efficiency,"KNN(k=5, metric=manhattan, voting=majority, retention=never_retain)",93.43% ± 0.78%,265.60,100.0%


## 4. Key Takeaways and Trade-Off Discussion

1. **Accuracy vs. Efficiency Trade-Off**:
   - On **Soybean**, the Canberra metric with `always_retain` achieves top accuracy (92.61%), but incurs longer prediction time (48.43 ms). Conversely, Euclidean metric with `never_retain` achieves 91.57% accuracy while reducing prediction time to 16.67 ms (a ~2.9x speedup with <1.1% accuracy degradation).
   - On **Hypothyroid**, Canberra with `always_retain` and inverse distance voting maximizes accuracy (96.63%), but requires 923.28 ms per fold due to instance accumulation and expensive per-feature scaling. In contrast, Manhattan metric with `never_retain` finishes prediction in 268.34 ms (~3.4x faster).
2. **Instance Retention Influence**:
   - `always_retain` continuously appends instances, expanding the case base. While this can provide a slight accuracy boost on clean datasets, it directly increases search latency in lazy learning inference.
   - For real-time deployment, models with `never_retain` or instance reduction represent practical operating points.
